In [4]:
!pip install opencv-python mediapipe

In [6]:
!pip install --upgrade mediapipe opencv-python

In [2]:
!pip install opencv-python urllib3

import os
import urllib.request

# Download Cascade XML files directly to ensure they exist locally
face_xml_url = 'https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml'
eye_xml_url = 'https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_eye.xml'

if not os.path.exists('haarcascade_frontalface_default.xml'):
  print('Downloading face cascade XML...')
  urllib.request.urlretrieve(face_xml_url, 'haarcascade_frontalface_default.xml')

if not os.path.exists('haarcascade_eye.xml'):
  print('Downloading eye cascade XML...')
  urllib.request.urlretrieve(eye_xml_url, 'haarcascade_eye.xml')

print('Setup complete! Both XML files are ready.')

Setup complete! Both XML files are ready.


In [1]:
import os
import cv2

# File paths for downloaded XMLs
face_cascade_path = 'haarcascade_frontalface_default.xml'
eye_cascade_path = 'haarcascade_eye.xml'

# Verify local files exist
if not os.path.exists(face_cascade_path) or not os.path.exists(
    eye_cascade_path
):
  raise FileNotFoundError('Please run Step 1 cell first to download XML files!')

# Load Cascade Classifiers from local XML files
face_cascade = cv2.CascadeClassifier(face_cascade_path)
eye_cascade = cv2.CascadeClassifier(eye_cascade_path)

# Initialize webcam
cap = cv2.VideoCapture(0)

print("Starting Face & Eye Tracking...")
print("Click on the webcam popup window and press 'q' to quit.")

while cap.isOpened():
  ret, frame = cap.read()
  if not ret:
    print('Failed to grab camera frame.')
    break

  # Mirror flip frame for natural view
  frame = cv2.flip(frame, 1)

  # Convert to Grayscale
  gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

  # Detect Faces
  faces = face_cascade.detectMultiScale(
      gray, scaleFactor=1.3, minNeighbors=5, minSize=(30, 30)
  )

  for x, y, w, h in faces:
    # Blue box around detected face
    cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
    cv2.putText(
        frame,
        'Recognized Face',
        (x, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 0, 0),
        2,
    )

    # Isolate Region of Interest (ROI) for eyes
    roi_gray = gray[y : y + h, x : x + w]
    roi_color = frame[y : y + h, x : x + w]

    # Detect Eyes inside face area
    eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=10)
    for ex, ey, ew, eh in eyes:
      # Green box around detected eyes
      cv2.rectangle(
          roi_color, (ex, ey), (ex + ew, ey + eh), (0, 255, 0), 2
      )
      cv2.putText(
          roi_color,
          'Eye',
          (ex, ey - 5),
          cv2.FONT_HERSHEY_SIMPLEX,
          0.4,
          (0, 255, 0),
          1,
      )

  # Display frame window
  cv2.imshow('Face Recognition & Eye Detection', frame)

  # Exit on pressing 'q'
  if cv2.waitKey(1) & 0xFF == ord('q'):
    break

# Safely close resources
cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)
print('Webcam feed closed successfully.')

Starting Face & Eye Tracking...
Click on the webcam popup window and press 'q' to quit.
Webcam feed closed successfully.
